# Test Multi-Agent System (MAS) Extraction

This notebook tests the full multi-agent orchestration pipeline with the `default` debate topology:
1. **Planner**: Generates a step-by-step extraction plan from the objective and document context.
2. **Players** (parallel): `value_identifier`, `labeller`, `schema_reasoner`, `critic`, `schema_expert` — 3 players per step, 2 debate rounds.
3. **Result**: Consolidated extraction records conforming to the `climate_vs_cropyield` schema, enforced by a Pydantic structured-output constraint on the final step.

In [1]:
import sys
sys.path.insert(0, '..')

from src.experimentutils import (
    get_all_markdown_paths,
    save_extraction_results_with_timestamp,
    ProgressOrchestrator,
)
from src.context import create_context
from src.standards import METADATA_STANDARDS
from src.core.schema_factory import SchemaFactory

In [2]:
# Get all paper paths
all_paths = get_all_markdown_paths()
print(f"Found {len(all_paths)} papers")

# Select first paper
sample_path = all_paths[0]
print(f"Testing with: {sample_path.split('/')[-1]}")

# Create execution context from the paper
context = create_context(source=sample_path, name="paper_to_extract")

Found 100 papers
Testing with: 1. Mason 1986 Cassava-cowpea and cassava-peanut intercropping II Leaf afrea index and dry matter accumulation.md


In [3]:
# Build the extraction objective and a Pydantic output schema from the same standard.
# The output_schema is passed to the orchestrator so the final synthesis step is
# forced to return a properly typed, schema-conforming Pydantic model.
standard = METADATA_STANDARDS["climate_vs_cropyield"]

OutputSchema = SchemaFactory().create_from_standard(
    standard,
    record_class_name="YieldRecord",
    output_class_name="MetaAnalysisOutput",
    records_key="yield_records",
)
print(f"Output schema: {OutputSchema.__name__}")
print(f"Record fields: {list(OutputSchema.model_fields['yield_records'].annotation.__args__[0].model_fields.keys())}")

Output schema: MetaAnalysisOutput
Record fields: ['crop_type', 'yield_value', 'location', 'year', 'yield_source_section', 'yield_confidence', 'yield_notes', 'Treatment', 'Treatment_source_section', 'Treatment_confidence', 'Treatment_notes', 'Tillage', 'Tillage_source_section', 'Tillage_confidence', 'Tillage_notes', 'Soil_property', 'Soil_source_section', 'Soil_confidence', 'Soil_notes', 'climate', 'climate_source_section', 'climate_confidence', 'climate_notes', 'remote_sensing_data', 'rs_source_section', 'rs_confidence', 'rs_notes']


In [4]:
objective = f'''You are an expert agricultural data extraction specialist. Your task is to extract **measured crop yield information** and related agronomic/context variables from scientific research papers. You must reason carefully and follow a multi-step extraction process to ensure accurate and complete data.

**META-ANALYTIC SCHEMA (CRITICAL)**:
{standard}

**SCHEMA HANDLING RULES**
- The meta-analytic schema above is expressed as JSON and/or `field: description` pairs.
- Treat the JSON keys / the text before the colon as the exact field names to use as keys in every yield record (e.g., `crop_type`, `yield_value`, `location`, `year`, `Treatment`, `Tillage`, `Soil_property`, `climate`, `remote_sensing_data`).
- When you construct any meta-analytic records (proposed, refined, or final), you MUST use these field names as the keys.
- The natural-language descriptions in the schema are only guidance and must **never** be copied as record values; every value must be the concrete value extracted from the paper.
- Do not introduce new top-level keys or rename existing ones in the final `yield_records`; use only the keys that appear in the schema.

**EXTRACTION PROCESS**

**Step 1: Label the Original Document**
The first step is to label the original document text by adding XML tags around schema-relevant values. This should be done by the labeller agent.

The labeller must read the complete original document from start to finish and add XML tags around schema-relevant text spans using schema field names (e.g., <crop_type>winter wheat</crop_type>, <yield_value>529.58 g/m2</yield_value>). The labeller must preserve ALL original text, formatting, paragraphs, and structure exactly as it appears. The labeller must NOT create summaries, lists, tables, or structured formats - only add tags to the original text. The output should be the complete original document with tags inserted inline, nothing else.

**Step 2: Extract and Structure Records**
After the document is labeled, extract structured records from the labeled text. For each identified yield anchor, extract all associated schema fields from the surrounding context (paragraphs, section headers, tables, captions). The schema fields to extract are:
- crop_type: Specific crop name (e.g., maize, wheat, rice, soybean, corn, etc.)
- yield_value: Actual yield measurement with units (e.g., 8.5 t/ha, 3500 kg/ha, 42 bu/acre)
- location: Study location (prefer coordinates if reported; otherwise country/province/state/city/research station/farm name)
- year: Data collection year or growing season (e.g., 2018, 2019-2020; not publication year)
- Treatment: The experimental treatment (e.g., Irrigation amount/frequency or total nitrogen fertilizer application amount and frequency)
- Tillage: Density and Planting (e.g., Row/plant spacing, density, etc.)
- Soil_property: Soil properties of the experimental sites (e.g., Texture, organic matter, pH, electrical conductivity, total nitrogen, available phosphorus/potassium, bulk density, field water holding capacity, available water capacity, and soil depth)
- climate: Climate data (e.g., daily/decadal Tmax/Tmin/Tmean, precipitation, radiation/sunshine, wind speed, VPD, ET0, GDD, KDD, cumulative precipitation, critical windows)
- remote_sensing_data: Raw spectral data or vegetation index (e.g., NDVI, EVI, GNDVI, NDRE, WDRVI, MCARI, red edge index, etc.)

Scan the entire content (including paragraphs, tables, captions, footnotes, and supplements) to locate all sentences or table entries that mention **actual measured crop yield values** (e.g., \"maize yield was 7.2 t/ha\"). Ignore model performance metrics (e.g., RMSE, R\u00b2), yield gaps, and simulated/modelled outputs.

**Complete Missing Fields and Validate**
If any required field for a yield record is **missing**, search specific related sections to retrieve it:
- For missing **location** or **year**, check **Materials and Methods / Site description**.
- For missing **crop type**, check **Abstract** and **Materials and Methods**.
- For missing **treatment (water/fertilizer)**, check **Field management / Experimental design / Plot / Block / Treatment / Table**.
- For missing **tillage (density & cultivation)**, check **Planting / Agronomic practices / Cultivation / Management**.
- For missing **soil properties**, check **Soil / Site description / Materials and Methods / Table**.
- For missing **climate**, check **Climate / Weather / Meteorology / Data sources**.
- For missing **UAV data**, check **UAV / UAS / Flight / Sensor / Data acquisition / Table / Figure captions**.

If you cannot find a field with **reasonable certainty**, discard the entire record. Only keep records where **all required fields are present and consistent**.

**Structure and Finalize Records**
For each complete and validated record, construct a structured JSON entry following the schema above. Keep original units and expressions; **do not convert**. If an associated field is not found with reasonable certainty, set its value to `null`. Apply de-duplication across records where (crop_type, yield_value+unit, location, year, treatment if any) are identical.

**TARGET DATA TO EXTRACT**:
- INCLUDE: Actual field-measured or reported yields (e.g., from experiments, harvest trials)
- EXCLUDE: Model evaluation metrics (RMSE, R2, MAE), yield gaps, predictions, correlation coefficients

**OUTPUT FORMAT**: Return results in JSON format with an array of yield records under the key `yield_records`. Each record's keys must be **exactly** the field names from the schema and the values must be the extracted data. If no yield data is found, return an empty array.'''

In [5]:
# Initialize orchestrator with the default debate topology
# default: 3 players per step, 2 debate rounds
orchestrator = ProgressOrchestrator(topology_name="default")

# Run the full pipeline: plan generation + execution (with live progress bars)
# output_schema forces the final synthesis step to return a schema-conforming Pydantic model
result = orchestrator.run(
    source=context,
    objective=objective,
    output_schema=OutputSchema,
)
print(f"Success: {result.success}")
print(f"Steps completed: {result.steps_completed}/{result.plan_steps_count}")

Generating plan:   0%|          | 0/1 [00:00<?, ?plan/s]

Executing plan:   0%|          | 0/4 [00:00<?, ?step/s]

  initializing:   0%|          | 0/6 [00:00<?, ?phase/s]

Success: True
Steps completed: 4/4


In [6]:
# Inspect step-by-step results
for step_result in result.step_results:
    print(f"Step {step_result.step_index + 1}: {step_result.task}")
    print(f"  Player: {step_result.player_role}")
    print(f"  Success: {step_result.success}")
    print(f"  Debate rounds: {step_result.debate_rounds_completed}")
    print(f"  Artifacts: {list(step_result.artifacts.keys())}")
    if step_result.error:
        print(f"  Error: {step_result.error}")
    print()

Step 1: Identify and list all field-value pairs from the document according to the meta-analytic schema.
  Player: value_identifier
  Success: True
  Debate rounds: 2
  Artifacts: ['field_value_pairs']

Step 2: Apply XML tags to the original document based on the identified field-value pairs.
  Player: labeller
  Success: True
  Debate rounds: 2
  Artifacts: ['labeled_text']

Step 3: Extract and structure complete meta-analysis records from the labeled text, ensuring all schema fields are populated or marked as null if not found.
  Player: schema_reasoner
  Success: True
  Debate rounds: 2
  Artifacts: ['proposed_meta_analysis_records']

Step 4: Validate and refine the proposed meta-analysis records against the schema, ensuring accuracy, completeness, and adherence to all rules.
  Player: schema_expert
  Success: True
  Debate rounds: 2
  Artifacts: ['final_meta_analysis_records']



In [7]:
# Display extracted records as a DataFrame
# With output_schema, the final artifact is a Pydantic model dumped to a dict:
# {"yield_records": [{field: value, ...}, ...]}
import pandas as pd

workspace = result.final_workspace
record_key = "final_meta_analysis_records"

raw = workspace.get(record_key, {})
if isinstance(raw, dict):
    records = raw.get("yield_records", raw.get("records", []))
elif isinstance(raw, list):
    records = raw
else:
    records = []

print(f"Extracted {len(records)} record(s)")
df = pd.DataFrame(records)
df[['crop_type', 'yield_value', 'location', 'year']]

Extracted 18 record(s)


,crop_type,yield_value,location,year
0,cassava,8 Mg ha⁻¹,"Santander de Quilichao, Colombia",1981
1,cowpea,2.3 Mg ha⁻¹,"Santander de Quilichao, Colombia",1981
2,peanut,4.4 Mg ha⁻¹,"Santander de Quilichao, Colombia",1981
3,cassava,7.5 Mg ha⁻¹,"Ibadan, Nigeria",1982
4,cassava,6.2 Mg ha⁻¹,"Ibadan, Nigeria",1982
5,cassava,5.8 Mg ha⁻¹,"Ibadan, Nigeria",1982
6,cowpea,3.9 Mg ha⁻¹,"Ibadan, Nigeria",1981
7,peanut,16.2 Mg ha⁻¹,"Ibadan, Nigeria",1981
8,cowpea,13.4 Mg ha⁻¹,"Ibadan, Nigeria",1982
9,peanut,18.5 Mg ha⁻¹,"Ibadan, Nigeria",1982


In [8]:
# Save results to a timestamped CSV — same pattern as test_direct_llm
saved_path = save_extraction_results_with_timestamp(
    results=raw,
    base_name="mas_extraction",
    records_key="yield_records",
    include_time=False,
)
print(f"Saved results to: {saved_path}")

# Verify
saved_df = pd.read_csv(saved_path)
print(f"Loaded {len(saved_df)} records from CSV")
saved_df.head()

Saved results to: /home/com3dian/Github/meta_analysis_agents/outputs/2026-03-11/mas_extraction_2026-03-11.csv
Loaded 18 records from CSV


,crop_type,yield_value,location,year,yield_source_section,yield_confidence,yield_notes,Treatment,Treatment_source_section,Treatment_confidence,...,Soil_confidence,Soil_notes,climate,climate_source_section,climate_confidence,climate_notes,remote_sensing_data,rs_source_section,rs_confidence,rs_notes
0,cassava,8 Mg ha⁻¹,"Santander de Quilichao, Colombia",1981,value_identifier_1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cowpea,2.3 Mg ha⁻¹,"Santander de Quilichao, Colombia",1981,value_identifier_1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,peanut,4.4 Mg ha⁻¹,"Santander de Quilichao, Colombia",1981,value_identifier_1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cassava,7.5 Mg ha⁻¹,"Ibadan, Nigeria",1982,labeller_2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cassava,6.2 Mg ha⁻¹,"Ibadan, Nigeria",1982,labeller_2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
